In [1]:
import pandas as pd
df = pd.read_pickle("dataset/dataset_v4_sig.pkl")

In [2]:
df.head()

,D2_daily_watch_time,A4_marital_status,B2_family_time_group,B3_family_relationship,B4_in_person_feeling,C2_weight_group,C3_sitting_duration,C4_physical_problems,D3_peak_watch_time,E2_sleep_trouble,...,G5_mental_stress,G6_panic_attacks,G8_addictive_watching,H1_mental_state_change,H2_content_type_effect,H3_distraction_after_viral,H4_brain_pressure,H5_sleep_quality_change,H6_emotional_reels_focus_loss,H7_curiosity_driven_rewatch
0,Less than 1 hour,Married,1-3 hours,Good,Somewhat Comfortable,50-60 kg,30 minutes–1 hour,Often,Before Sleeping,No,...,Yes,Sometimes,Sometimes,Some Change,Negative Impact,Sometimes,Moderate Stress,Slight Change in Sleep Quality,Lose Focus Frequently,Sometimes Interested
1,Less than 1 hour,Unmarried,1-3 hours,Very Good,Neutral,Above 80 kg,Less than 30 minutes,Never,Before Sleeping,No,...,No,Never,Never,No Change,No Impact,Never,No Stress,Not Sure,Do Not Lose Focus,Rarely Interested
2,More than that,Unmarried,5-8 hours,Very Good,Somewhat Comfortable,70-80 kg,More than 2 hours,Sometimes,Before Sleeping,Sometimes,...,Yes,Sometimes,Always,Some Change,Not Sure,Sometimes,No Stress,Not Sure,Lose Focus Sometimes,More Interest in Watching Reels
3,1 hour,Married,1-3 hours,Moderate,Very Comfortable,70-80 kg,30 minutes–1 hour,Sometimes,Leisure Time,Sometimes,...,No,Never,Sometimes,Some Change,Not Sure,Never,No Stress,Not Sure,Do Not Lose Focus,Sometimes Interested
4,2 hours,Unmarried,Up to 1 hour,Good,Neutral,Up to 50 kg,1–2 hours,Sometimes,Leisure Time,Yes,...,Yes,Sometimes,Always,Some Change,Not Sure,Rarely,Low Stress,Slight Change in Sleep Quality,Lose Focus Rarely,Sometimes Interested


In [3]:
import warnings
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2

OUT = 'D2_daily_watch_time'
ORDER = ['Less than 1 hour', '1 hour', '2 hours', '3 hours', 'More than that']
ALPHA = 0.05
fmt_p = lambda p: '< 0.0001' if p < 0.0001 else f'{p:.4f}'

# Outcome coded 0-4; the first category ("Less than 1 hour") is the reference
df[OUT] = pd.Categorical(df[OUT], categories=ORDER, ordered=True)
y = pd.Series(df[OUT].cat.codes, name=OUT)
preds = [c for c in df.columns if c != OUT]

# Each predictor is dummy-coded; its most frequent category is the reference
refs = {c: df[c].value_counts().idxmax() for c in preds}


def design(cols):
    parts = [pd.get_dummies(df[c], prefix=c, prefix_sep=': ', dtype=float).drop(columns=f'{c}: {refs[c]}')
             for c in cols]
    return sm.add_constant(pd.concat(parts, axis=1))


def fit(X):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return sm.MNLogit(y, X).fit(method='bfgs', maxiter=5000, disp=0)


X = design(preds)
model = fit(X)
print(f'Observations: {len(y)} | predictors: {len(preds)} | dummy columns: {X.shape[1] - 1} | parameters: {model.params.size}')
print('Converged:', model.mle_retvals['converged'])
pd.Series(refs, name='Reference category').to_frame()

Observations: 503 | predictors: 28 | dummy columns: 86 | parameters: 348
Converged: True


,Reference category
A4_marital_status,Unmarried
B2_family_time_group,1-3 hours
B3_family_relationship,Very Good
B4_in_person_feeling,Very Comfortable
C2_weight_group,60-70 kg
C3_sitting_duration,30 minutes–1 hour
C4_physical_problems,Sometimes
D3_peak_watch_time,Before Sleeping
E2_sleep_trouble,No
E3_exercise_involvement,Sometimes


In [4]:
# Model fitting information and pseudo R-square
n, ll, ll0 = len(y), model.llf, model.llnull
cox_snell = 1 - np.exp(2 * (ll0 - ll) / n)
nagelkerke = cox_snell / (1 - np.exp(2 * ll0 / n))

fit_info = pd.DataFrame({'Value': [
    f'{-2 * ll0:.2f}', f'{-2 * ll:.2f}', f'{model.llr:.2f}', f'{model.df_model:.0f}', fmt_p(model.llr_pvalue),
    f'{model.aic:.2f}', f'{model.bic:.2f}', f'{cox_snell:.3f}', f'{nagelkerke:.3f}', f'{model.prsquared:.3f}']},
    index=['-2 Log Likelihood (intercept only)', '-2 Log Likelihood (final)', 'Likelihood ratio chi-square', 'df',
           'p-value', 'AIC', 'BIC', 'Cox & Snell R²', 'Nagelkerke R²', 'McFadden R²'])
fit_info

,Value
-2 Log Likelihood (intercept only),1586.36
-2 Log Likelihood (final),809.83
Likelihood ratio chi-square,776.53
df,344
p-value,< 0.0001
AIC,1505.83
BIC,2974.60
Cox & Snell R²,0.786
Nagelkerke R²,0.822
McFadden R²,0.490


In [5]:
# Likelihood ratio test per predictor: final model vs. the model without that predictor
rows = []
for c in preds:
    reduced = fit(design([p for p in preds if p != c]))
    stat = max(2 * (model.llf - reduced.llf), 0)
    dof = model.df_model - reduced.df_model
    rows.append((c, stat, dof, chi2.sf(stat, dof)))

lr_tests = pd.DataFrame(rows, columns=['Variable', 'LR chi-square', 'df', 'p-value'])
lr_tests['Significant'] = lr_tests['p-value'] < ALPHA
lr_tests.round({'LR chi-square': 2}).assign(**{'p-value': lambda t: t['p-value'].map(fmt_p)})

,Variable,LR chi-square,df,p-value,Significant
0,A4_marital_status,20.01,8.0,0.0103,True
1,B2_family_time_group,18.30,16.0,0.3066,False
2,B3_family_relationship,9.97,12.0,0.6185,False
3,B4_in_person_feeling,51.26,16.0,< 0.0001,True
4,C2_weight_group,49.19,16.0,< 0.0001,True
5,C3_sitting_duration,206.34,12.0,< 0.0001,True
6,C4_physical_problems,15.35,12.0,0.2226,False
7,D3_peak_watch_time,34.65,20.0,0.0221,True
8,E2_sleep_trouble,10.74,8.0,0.2168,False
9,E3_exercise_involvement,20.45,12.0,0.0590,False


In [6]:
# Parameter estimates: each outcome category vs. "Less than 1 hour"
parts = []
with np.errstate(over='ignore'):
    for j, name in enumerate(ORDER[1:]):
        b, se, p = model.params.iloc[:, j], model.bse.iloc[:, j], model.pvalues.iloc[:, j]
        parts.append(pd.DataFrame({
            'Outcome': f'{name} vs {ORDER[0]}', 'Term': b.index, 'B': b.values, 'SE': se.values,
            'Wald': (b / se).values ** 2, 'p-value': p.values, 'Exp(B)': np.exp(b).values,
            '95% CI lower': np.exp(b - 1.96 * se).values, '95% CI upper': np.exp(b + 1.96 * se).values}))
estimates = pd.concat(parts, ignore_index=True)

# Very large SEs mean (quasi-)separation: those odds ratios are not reliable
estimates['Unstable'] = estimates['SE'] > 10
print(f"Unstable estimates (SE > 10): {estimates['Unstable'].sum()} of {len(estimates)}")
estimates.round(3).assign(**{'p-value': lambda t: t['p-value'].map(fmt_p)})

Unstable estimates (SE > 10): 11 of 348


,Outcome,Term,B,SE,Wald,p-value,Exp(B),95% CI lower,95% CI upper,Unstable
0,1 hour vs Less than 1 hour,const,4.645,1.654,7.889,0.0050,104.112,4.070,2663.118,False
1,1 hour vs Less than 1 hour,A4_marital_status: Divorced,-20.521,1506.665,0.000,0.9890,0.000,0.000,inf,True
2,1 hour vs Less than 1 hour,A4_marital_status: Married,-1.111,0.605,3.371,0.0660,0.329,0.101,1.078,False
3,1 hour vs Less than 1 hour,B2_family_time_group: Up to 1 hour,0.709,0.993,0.510,0.4750,2.032,0.290,14.225,False
4,1 hour vs Less than 1 hour,B2_family_time_group: 3-5 hours,-0.248,0.766,0.105,0.7460,0.781,0.174,3.500,False
...,...,...,...,...,...,...,...,...,...,...
343,More than that vs Less than 1 hour,H6_emotional_reels_focus_loss: Lose Focus Freq...,-2.249,1.155,3.794,0.0510,0.106,0.011,1.014,False
344,More than that vs Less than 1 hour,H6_emotional_reels_focus_loss: Lose Focus Rarely,-2.629,1.002,6.882,0.0090,0.072,0.010,0.514,False
345,More than that vs Less than 1 hour,H7_curiosity_driven_rewatch: More Interest in ...,-1.268,0.921,1.893,0.1690,0.281,0.046,1.713,False
346,More than that vs Less than 1 hour,H7_curiosity_driven_rewatch: Not Interested,-3.463,1.037,11.150,0.0010,0.031,0.004,0.239,False


In [7]:
# Significant terms only (p < 0.05)
sig_terms = estimates[(estimates['p-value'] < ALPHA) & (estimates['Term'] != 'const')]
sig_terms.round(3).assign(**{'p-value': lambda t: t['p-value'].map(fmt_p)})

,Outcome,Term,B,SE,Wald,p-value,Exp(B),95% CI lower,95% CI upper,Unstable
10,1 hour vs Less than 1 hour,B4_in_person_feeling: Neutral,-2.963,0.714,17.221,< 0.0001,0.052,0.013,0.209,False
11,1 hour vs Less than 1 hour,B4_in_person_feeling: Somewhat Comfortable,-2.349,0.735,10.209,0.0010,0.095,0.023,0.403,False
15,1 hour vs Less than 1 hour,C2_weight_group: 50-60 kg,-1.551,0.666,5.414,0.0200,0.212,0.057,0.783,False
18,1 hour vs Less than 1 hour,C3_sitting_duration: 1–2 hours,2.477,0.918,7.275,0.0070,11.908,1.968,72.053,False
19,1 hour vs Less than 1 hour,C3_sitting_duration: Less than 30 minutes,-2.419,0.555,19.020,< 0.0001,0.089,0.030,0.264,False
...,...,...,...,...,...,...,...,...,...,...
332,More than that vs Less than 1 hour,H2_content_type_effect: Positive Impact,-2.564,1.006,6.500,0.0110,0.077,0.011,0.553,False
342,More than that vs Less than 1 hour,H6_emotional_reels_focus_loss: Do Not Lose Focus,-3.158,1.057,8.933,0.0030,0.043,0.005,0.337,False
344,More than that vs Less than 1 hour,H6_emotional_reels_focus_loss: Lose Focus Rarely,-2.629,1.002,6.882,0.0090,0.072,0.010,0.514,False
346,More than that vs Less than 1 hour,H7_curiosity_driven_rewatch: Not Interested,-3.463,1.037,11.150,0.0010,0.031,0.004,0.239,False


In [8]:
# Classification table: observed vs. predicted (highest predicted probability)
predicted = model.predict(X).values.argmax(axis=1)
classification = pd.crosstab(pd.Categorical.from_codes(y, ORDER), pd.Categorical.from_codes(predicted, ORDER),
                             rownames=['Observed'], colnames=['Predicted'])
classification['% correct'] = (np.diag(classification[ORDER]) / classification[ORDER].sum(axis=1) * 100).round(1)
print(f'Overall % correct: {(predicted == y).mean() * 100:.1f}%')
classification

Overall % correct: 67.2%


Predicted,Less than 1 hour,1 hour,2 hours,3 hours,More than that,% correct
Observed,,,,,,
Less than 1 hour,88,9,7,2,2,81.5
1 hour,16,64,22,2,4,59.3
2 hours,9,17,88,7,13,65.7
3 hours,2,5,13,30,8,51.7
More than that,3,5,15,4,68,71.6


In [9]:
# PDF report: odds ratios per outcome category vs. "Less than 1 hour"
import os
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer

PDF_PATH = os.path.join('mlr_analysis', 'mlr_D2_daily_watch_time.pdf')
GROUPS = ORDER[1:]
NCOL = 1 + 4 * len(GROUPS)
fmt_p2 = lambda p: '<0.01' if p < 0.01 else f'{p:.2f}'
label = lambda c: c.split('_', 1)[1].replace('_', ' ').capitalize() + f" ({c.split('_', 1)[0]})"

body = ParagraphStyle('body', fontName='Times-Roman', fontSize=7.5, leading=9, alignment=1)
bold = ParagraphStyle('bold', parent=body, fontName='Times-Bold')
left = ParagraphStyle('left', parent=body, alignment=0)
note = ParagraphStyle('note', parent=body, fontSize=8, leading=10, alignment=0)
title = ParagraphStyle('title', parent=body, fontName='Times-Bold', fontSize=12, leading=15)

# Three header rows: outcome, outcome categories, statistics
rows = [
    [Paragraph('Variables', bold), Paragraph(f'Daily watch time (Ref: {ORDER[0]}/day)', bold)] + [''] * (NCOL - 2),
    [''] + sum([[Paragraph(g, bold), '', '', ''] for g in GROUPS], []),
    [''] + [Paragraph(h, bold) for h in ['P-value', 'OR', '95% CI Lower', '95% CI Upper']] * len(GROUPS),
]
style = [
    ('SPAN', (0, 0), (0, 2)), ('SPAN', (1, 0), (-1, 0)),
    *[('SPAN', (1 + 4 * j, 1), (4 + 4 * j, 1)) for j in range(len(GROUPS))],
    ('GRID', (0, 0), (-1, -1), 0.5, colors.black), ('BOX', (0, 0), (-1, -1), 1, colors.black),
    ('FONTNAME', (0, 0), (-1, -1), 'Times-Roman'), ('FONTSIZE', (0, 0), (-1, -1), 7.5),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'), ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
]

# One section row per predictor, then one row per non-reference category
est = estimates.set_index(['Outcome', 'Term'])
for c in preds:
    rows.append([Paragraph(f'{label(c)} (Ref: {refs[c]})', bold)] + [''] * (NCOL - 1))
    style.append(('SPAN', (0, len(rows) - 1), (-1, len(rows) - 1)))
    for cat in df[c].cat.categories:
        if cat == refs[c]:
            continue
        row = [Paragraph(str(cat), left)]
        for g in GROUPS:
            r = est.loc[(f'{g} vs {ORDER[0]}', f'{c}: {cat}')]
            if r['Unstable']:
                row += [fmt_p2(r['p-value']), 'NE\u2020', '\u2013', '\u2013']
            else:
                row += [fmt_p2(r['p-value'])] + [f'{r[k]:.3f}' for k in ['Exp(B)', '95% CI lower', '95% CI upper']]
        rows.append(row)

width = landscape(A4)[0] - 60
var_w = 150
table = Table(rows, colWidths=[var_w] + [(width - var_w) / (NCOL - 1)] * (NCOL - 1), repeatRows=3)
table.setStyle(TableStyle(style))

notes = (f'OR = odds ratio; CI = confidence interval. Multinomial logistic regression (N = {len(y)}); '
         f'reference outcome category: {ORDER[0]}/day. Predictors: the {len(preds)} variables significantly associated '
         f'with daily watch time (chi-square / Fisher-Freeman-Halton, p &lt; {ALPHA}). '
         f'Model fit: likelihood ratio chi-square = {model.llr:.2f}, df = {model.df_model:.0f}, p {"< 0.01" if model.llr_pvalue < 0.01 else "= " + fmt_p2(model.llr_pvalue)}; '
         f'Nagelkerke R<super>2</super> = {nagelkerke:.3f}; McFadden R<super>2</super> = {model.prsquared:.3f}; '
         f'overall classification accuracy = {(predicted == y).mean() * 100:.1f}%.<br/>'
         '\u2020 NE = not estimable: SE &gt; 10 because of very few respondents in that category (quasi-complete separation).')

os.makedirs(os.path.dirname(PDF_PATH), exist_ok=True)
SimpleDocTemplate(PDF_PATH, pagesize=landscape(A4), leftMargin=30, rightMargin=30, topMargin=30, bottomMargin=30).build([
    Paragraph('Multinomial logistic regression: factors associated with daily Reels watch time', title),
    Spacer(1, 8), table, Spacer(1, 8), Paragraph(notes, note)])
print('Saved', PDF_PATH)

Saved mlr_analysis/mlr_D2_daily_watch_time.pdf
